# Feature Engineering & Preprocessing Pipeline
## Leak-free Scikit-learn workflow

**Dataset:** `customer_risk_synthetic.csv` (500 synthetic customer records). This is an educational synthetic dataset, not real customer information.

### Objectives
- Explore a mixed numerical/categorical tabular dataset.
- Split into training and test sets **before fitting any preprocessing steps**.
- Impute missing values, scale numeric columns, and one-hot encode categorical columns using a reusable `ColumnTransformer` + `Pipeline`.
- Conduct correlation analysis and feature-importance analysis.
- Keep identifiers out of model features and avoid target leakage.

> Run the notebook from top to bottom. Upload `customer_risk_synthetic.csv` when prompted if it is not already in the Colab session.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score


## 2. Load the dataset

In [ ]:
# In Google Colab, choose the CSV file you downloaded.
try:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(iter(uploaded))
except ImportError:
    # If running locally, place the CSV in the same folder as this notebook.
    csv_name = "customer_risk_synthetic.csv"

df = pd.read_csv(csv_name)
print("Dataset shape:", df.shape)
display(df.head())


## 3. Explore and check data quality

In [ ]:
print("Column types and non-null counts:")
df.info()

print("\\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count"))

print("\\nTarget distribution:")
display(df["churned"].value_counts().rename_axis("churned").to_frame("count"))

print("\\nSummary statistics:")
display(df.describe(include="all").T)


## 4. Separate target and features, then split first

`customer_id` is an identifier, so it is removed from predictors. The target `churned` is kept separate. The train/test split happens **before** the imputer, scaler, encoder, or model is fitted. This prevents information from the test set leaking into preprocessing.

In [ ]:
target = "churned"
drop_columns = ["customer_id", target]

X = df.drop(columns=drop_columns)
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion"))


## 5. Identify numerical and categorical columns

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


## 6. Build the reusable preprocessing + modelling pipeline

- Numerical columns: median imputation, then standard scaling.
- Categorical columns: most-frequent imputation, then one-hot encoding.
- `handle_unknown="ignore"` lets the pipeline handle new categories at prediction time.
- The transformations are fitted only on the training set when `pipeline.fit()` runs.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ))
])

pipeline


## 7. Fit on training data and evaluate on held-out test data

In [ ]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, zero_division=0))
if y_test.nunique() == 2:
    print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Test-set confusion matrix")
plt.show()


## 8. Correlation analysis

Correlation is calculated on numerical predictor columns only. The target is not used to fit preprocessing. Correlation shows pairwise linear relationships; it does not prove causation.

In [ ]:
numeric_for_corr = df[numeric_features].copy()
corr = numeric_for_corr.corr(numeric_only=True)

display(corr.round(2))
plt.figure(figsize=(8, 5))
plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation matrix: numerical predictors")
plt.tight_layout()
plt.show()


## 9. Feature importance using mutual information

Mutual information is calculated on the training data after preprocessing. The fitted preprocessor transforms training features; the feature names are taken from the fitted transformer. This is a feature-ranking exercise, not proof of causal influence.

In [ ]:
X_train_transformed = pipeline.named_steps["preprocessor"].transform(X_train)
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

mi_scores = mutual_info_classif(
    X_train_transformed, y_train, random_state=42
)
mi_table = pd.DataFrame({
    "feature": feature_names,
    "mutual_information": mi_scores
}).sort_values("mutual_information", ascending=False)

display(mi_table.head(15))

top_mi = mi_table.head(15).sort_values("mutual_information")
plt.figure(figsize=(9, 6))
plt.barh(top_mi["feature"], top_mi["mutual_information"])
plt.xlabel("Mutual information")
plt.title("Top features by mutual information (training set)")
plt.tight_layout()
plt.show()


## 10. Tree-based feature importance

In [ ]:
tree_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": pipeline.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

display(tree_importance.head(15))

top_tree = tree_importance.head(15).sort_values("importance")
plt.figure(figsize=(9, 6))
plt.barh(top_tree["feature"], top_tree["importance"])
plt.xlabel("Random Forest importance")
plt.title("Top features by tree-based importance")
plt.tight_layout()
plt.show()


## 11. Demonstrate reuse on new records

Use the original feature columns (without `customer_id` and `churned`). The same fitted pipeline automatically applies the same imputation, scaling, and encoding rules.

In [ ]:
new_customers = X_test.head(3).copy()
predictions = pipeline.predict(new_customers)
probabilities = pipeline.predict_proba(new_customers)[:, 1]

preview = new_customers.copy()
preview["predicted_churn"] = predictions
preview["predicted_churn_probability"] = probabilities.round(3)
display(preview)


## 12. Final checklist and conclusion

- [x] Mixed numerical and categorical tabular data loaded.
- [x] Train/test split performed before fitting preprocessing.
- [x] Missing numerical and categorical values handled by imputers.
- [x] Numerical features scaled.
- [x] Categorical features one-hot encoded.
- [x] Reusable Scikit-learn `Pipeline` and `ColumnTransformer` created.
- [x] Correlation analysis performed.
- [x] Mutual-information and tree-based feature rankings produced.
- [x] Pipeline reused for new records.

**Conclusion:** This notebook demonstrates a reusable, leak-free preprocessing workflow. The dataset is synthetic and intended for learning; model scores should not be interpreted as real-world customer churn performance.
